> ⚠ **Not migrated: depends on the v1 YAML parser-options /
> thermal-validation API (v2 gap, migration-from-v1.md §7).**
>
> This tutorial is entirely about `ps.YamlParserOptions(strict=...)`
> and the parser's thermal-block validation diagnostics. v2 removed
> that surface:
>
> - `ps.YamlParserOptions` — **no v2 equivalent**;
>   `p.load_yaml_file` / `p.load_yaml_string` take no options
>   (deprecation map in `pulsim/__init__.py`).
> - `ps.YamlParser(...).load_string(...)` returning `(circuit, sim_opts)`
>   with `parser.errors` / `parser.warnings` — replaced by
>   `p.load_yaml_string(text)` returning `LoadedCircuit(builder, options)`,
>   which raises on malformed input instead of accumulating diagnostics.
> - The v1 `schema: pulsim-v1` thermal block and its diagnostics
>   (`PULSIM_YAML_E_THERMAL_MISSING_REQUIRED`,
>   `PULSIM_YAML_W_THERMAL_DEFAULT_APPLIED`,
>   `PULSIM_YAML_E_THERMAL_UNSUPPORTED_COMPONENT`) — the v2 loader uses
>   a different schema (top-level `circuit:`) with **no `thermal:` block**.
> - `options.thermal_devices` — not present in v2 `SimulationOptions`.
>
> There is no drop-in v2 replacement for strict-vs-non-strict thermal
> validation, so the cells below are left inert and the notebook
> executes without error.

# Tutorial: Electrothermal YAML Validation Modes (v1 — NOT migrated)

**Original audience:** users configuring thermal ports and parser
validation in CI.

**Original learning goals (v1 API, no v2 equivalent):**

- strict vs non-strict behavior for missing thermal constants,
- diagnostics for unsupported thermal-capable components,
- how to inspect parser errors and warnings programmatically.


## Outline

1. Build a minimal helper to parse YAML strings.
2. Strict mode: missing `rth`/`cth` should fail.
3. Non-strict mode: defaults should be applied with warnings.
4. Unsupported component thermal enablement should fail deterministically.


In [1]:
# v2 import works fine; the gap is the YAML parser-options /
# thermal-validation API this tutorial was built on.
import pulsim as p

print("Pulsim:", p.__version__)
print()
print("This tutorial is NOT migrated to v2. It exercised")
print("ps.YamlParserOptions(strict=...) and parser thermal diagnostics,")
print("which have no v2 equivalent. v2 loaders raise on bad input")
print("instead of accumulating errors/warnings. See the top banner.")

# Confirm the v1 symbols this tutorial depends on are absent in v2:
missing = [name for name in ("YamlParser", "YamlParserOptions")
           if not hasattr(p, name)]
print()
print("v1 symbols absent in v2:", missing)
assert missing == ["YamlParser", "YamlParserOptions"]


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Pulsim: 1.6.4

This tutorial is NOT migrated to v2. It exercised
ps.YamlParserOptions(strict=...) and parser thermal diagnostics,
which have no v2 equivalent. v2 loaders raise on bad input
instead of accumulating errors/warnings. See the top banner.

v1 symbols absent in v2: ['YamlParser', 'YamlParserOptions']


## Step 1 - Compare parser modes (v1 flow — reference only)

In v1 the same incomplete thermal block behaved differently by parser
mode (strict = error, non-strict = defaults + warning). v2 has no
`YamlParserOptions` and no thermal-block diagnostics, so the code below
is shown for reference only and is not executed.


### Original v1 code (reference only — does not run on v2)

The original implementation built incomplete-thermal YAML strings and
parsed them with `ps.YamlParser` in strict and non-strict mode,
asserting on specific diagnostic codes. None of that API exists in v2,
so the block is shown for reference only and is not executed.

```python
# v1 helper
def parse_string(yaml_text: str, strict: bool):
    opts = ps.YamlParserOptions()
    opts.strict = strict
    parser = ps.YamlParser(opts)
    circuit, sim_opts = parser.load_string(yaml_text)
    return parser, circuit, sim_opts

missing_constants_yaml = """
schema: pulsim-v1
version: 1
simulation:
  tstop: 1e-4
  dt: 1e-6
  thermal:
    default_rth: 1.7
    default_cth: 0.3
components:
  - type: mosfet
    name: M1
    nodes: [gate, drain, source]
    thermal:
      enabled: true
"""

strict_parser, _, _ = parse_string(missing_constants_yaml, strict=True)
# strict mode -> PULSIM_YAML_E_THERMAL_MISSING_REQUIRED in parser.errors

non_strict_parser, _, non_strict_opts = parse_string(missing_constants_yaml, strict=False)
# non-strict mode -> PULSIM_YAML_W_THERMAL_DEFAULT_APPLIED in parser.warnings,
# defaults visible via non_strict_opts.thermal_devices["M1"].rth / .cth

# A thermal-enabled resistor -> PULSIM_YAML_E_THERMAL_UNSUPPORTED_COMPONENT
```

**v2 note.** The v2 loaders (`p.load_yaml_file`, `p.load_yaml_string`)
parse a different schema (top-level `circuit:`) with no `thermal:`
block and no parser options; malformed input raises a `RuntimeError`
rather than collecting per-diagnostic codes. A v2 thermal YAML schema
plus structured diagnostics is future work.


## Exercises

1. Add `rth` but keep `cth` missing in strict mode. Confirm the exact diagnostic path.
2. Set `rth: 0` and `cth: -1` and inspect range diagnostics.
3. Try a `bjt_npn` with thermal enabled and verify it is accepted.


In [2]:
# Exercise answer scaffold
# Build your YAML strings and call parse_string(yaml_text, strict=True/False).
# Then print parser.errors / parser.warnings and inspect options.thermal_devices.

# TODO: implement your experiment here.
